# **Transformer Interpretability**

In this coding homework, you will:
* Implement a single attention head
* Implement an induction copy head by combining a previous token head with a copying head.

In [ ]:
#pull the github repo
!git clone https://github.com/Berkeley-CS182/cs182fa25_public.git

In [ ]:
%cd cs182fa25_public/hw11/code/q_coding_interpretability/

## **Imports & Preliminaries**

In [ ]:
"""Transformer Interpretability Module.

This module implements attention mechanisms for understanding
transformer model interpretability, including single attention heads
and induction copy heads.

References:
    - Elhage et al. (2021): "A Mathematical Framework for Transformer Circuits"
    - Olsson et al. (2022): "In-context Learning and Induction Heads"
"""
from __future__ import annotations

from typing import Union, Sequence
import json

import numpy as np
from numpy.typing import ArrayLike, NDArray

# Type aliases for clearer function signatures
FloatArray = NDArray[np.floating]
Matrix = Union[Sequence[Sequence[float]], FloatArray]

# Configuration constants
RANDOM_SEED: int = 2025
NUMERICAL_TOLERANCE: float = 1e-6
MASK_VALUE: float = -1e9  # Large negative value for causal masking

# Set reproducible random state
np.random.seed(RANDOM_SEED)

## **Question 1: A Single Attention Head**
In this question, you'll implement an attention head.

### Specifications:
* The query-key matrices are provided already multiplied together (i.e., we provide as input $W_{QK} = W_Q^\top W_K$, called `WQK`).
* The output-value matrices are provided already multiplied together (i.e., we provide as input $W_{OV} = W_O^\top W_V$, called `WOV`).
* Attention inputs are also provided as a list of `d_model`-length vectors, called `attn_input`. The list elements correspond to positions in the context.
* The desired outputs are the outputs the attention head produces at each position. This should be a list of `d-model`-length vectors of the same length as the input.
* **Causal masking:** When implementing attention, mask out positions that come after the current position by setting their attention scores to negative infinity (e.g., `-1e9`) before applying softmax.
* You should first convert the inputs to numpy arrays as a first step.

### Note:
* You should **not** use `np.softmax` when calculating the attention scores.

In [ ]:
def compute_softmax(logits: FloatArray, axis: int = -1) -> FloatArray:
    """Compute numerically stable softmax along specified axis.

    Uses the log-sum-exp trick to prevent overflow by subtracting
    the maximum value before exponentiation.

    Parameters
    ----------
    logits : FloatArray
        Input array of logits (unnormalized log probabilities).
    axis : int, optional
        Axis along which to compute softmax, by default -1.

    Returns
    -------
    FloatArray
        Probability distribution that sums to 1 along the specified axis.

    Notes
    -----
    The softmax function is defined as:
        softmax(x_i) = exp(x_i) / sum(exp(x_j))

    For numerical stability, we compute:
        softmax(x_i) = exp(x_i - max(x)) / sum(exp(x_j - max(x)))

    Examples
    --------
    >>> logits = np.array([1.0, 2.0, 3.0])
    >>> probs = compute_softmax(logits)
    >>> np.isclose(probs.sum(), 1.0)
    True
    """
    # Subtract max for numerical stability (log-sum-exp trick)
    shifted_logits = logits - np.max(logits, axis=axis, keepdims=True)
    exp_logits = np.exp(shifted_logits)
    return exp_logits / np.sum(exp_logits, axis=axis, keepdims=True)


def create_causal_mask(seq_len: int) -> NDArray[np.bool_]:
    """Create a causal (lower-triangular) attention mask.

    In causal attention, each position can only attend to itself
    and earlier positions, not future positions.

    Parameters
    ----------
    seq_len : int
        Length of the sequence.

    Returns
    -------
    NDArray[np.bool_]
        Boolean mask of shape (seq_len, seq_len) where True indicates
        positions that should be masked (set to -inf before softmax).

    Examples
    --------
    >>> create_causal_mask(3)
    array([[False,  True,  True],
           [False, False,  True],
           [False, False, False]])
    """
    return np.triu(np.ones((seq_len, seq_len), dtype=bool), k=1)

In [ ]:
def single_attention_head(
    attn_input: Matrix,
    w_qk: Matrix,
    w_ov: Matrix,
) -> FloatArray:
    """Implement a single causal attention head.

    This function computes scaled dot-product attention with causal masking.
    The attention mechanism allows each position to attend to itself and
    all previous positions, but not future positions.

    Parameters
    ----------
    attn_input : Matrix
        Input embeddings of shape (seq_len, d_model).
        Each row represents a token's embedding vector.
    w_qk : Matrix
        Pre-multiplied query-key weight matrix W_Q^T @ W_K of shape (d_model, d_model).
        This combines the query and key projections.
    w_ov : Matrix
        Pre-multiplied output-value weight matrix W_O^T @ W_V of shape (d_model, d_model).
        This combines the value and output projections.

    Returns
    -------
    FloatArray
        Output of the attention head, shape (seq_len, d_model).

    Notes
    -----
    The attention computation follows these steps:

    1. Compute attention scores: A = X @ W_QK @ X^T
    2. Apply causal mask: A[future_positions] = -inf
    3. Apply softmax: P = softmax(A, axis=-1)
    4. Compute output: O = P @ (X @ W_OV^T)

    The causal mask ensures autoregressive behavior where token i
    can only attend to tokens 0, 1, ..., i.

    Examples
    --------
    >>> x = [[0, 1], [1, 1], [1, 2]]
    >>> w_qk = [[1, 1], [0, 0]]
    >>> w_ov = [[1, 1], [0, 0]]
    >>> output = single_attention_head(x, w_qk, w_ov)
    >>> output.shape
    (3, 2)
    """
    # Convert inputs to numpy arrays (avoid shadowing parameter names)
    input_embeddings = np.asarray(attn_input, dtype=np.float64)
    query_key_weights = np.asarray(w_qk, dtype=np.float64)
    output_value_weights = np.asarray(w_ov, dtype=np.float64)

    seq_len, _ = input_embeddings.shape

    # Step 1: Compute pre-softmax attention scores
    # Score[i,j] = query_i @ key_j = x_i @ W_QK @ x_j^T
    attention_logits = input_embeddings @ query_key_weights @ input_embeddings.T

    # Step 2: Apply causal mask (prevent attending to future positions)
    causal_mask = create_causal_mask(seq_len)
    attention_logits[causal_mask] = MASK_VALUE

    # Step 3: Convert logits to probabilities via softmax
    attention_weights = compute_softmax(attention_logits, axis=1)

    # Step 4: Compute weighted combination of value projections
    value_projections = input_embeddings @ output_value_weights.T
    output = attention_weights @ value_projections

    return output

### Test `single_attention_head` basic

In [ ]:
attn_input = [[0, 1],
              [1, 1],
              [1, 2]]
WQK = [[1, 1],
       [0, 0]]
WOV = [[1, 1],
       [0, 0]]

expected_out = [[1.0, 0.0],
                [1.7310585786300048, 0.0],
                [2.5752103826044417, 0.0]]

out = single_attention_head(attn_input, WQK, WOV).tolist()

assert np.isclose(expected_out, out, atol=NUMERICAL_TOLERANCE).all(), f"Failed:\nExpected: {expected_out}\nGot: {out}"
print("Test case passed ✅")

### Test `single_attention_head`

In [ ]:
with open('single_attention_head_test_cases.json', 'r') as f:
    single_attention_head_test_cases = json.load(f)

for test_case_id in single_attention_head_test_cases:
    attn_input, WQK, WOV, expected_out = single_attention_head_test_cases[test_case_id]
    out = single_attention_head(attn_input, WQK, WOV).tolist()
    test_id = test_case_id.split(" ")[-1]
    assert np.isclose(expected_out, out, atol=NUMERICAL_TOLERANCE).all(), f"Test Case {test_id} Failed:\nExpected: {expected_out}\nGot: {out}"

print("All test cases passed ✅")

## **Question 2: An Induction Copy Head**
In this problem, you will combine a previous token head with a copying head.

### Background:
An induction head operates by predicting that previously-seen adjacencies in the sequence will be seen again. That is, it predicts `ab...a` will be followed by `b` (for any `a, b`).

### Specifications:
* #### Vocabulary and Embeddings
  - Vocabulary size: 4
  - Tokens: `a`, `b`, `c`, `d`
  - Maximum sequence length: 5
  - Embedding: 2-hot encoded with `d_model = 9`
    - First 4 dimensions: 1-hot encoding of token vocabulary
    - Next 5 dimensions: 1-hot encoding of position (0-4)
  - The unembeddings are the same as the embeddings. The model produces an output vector of length 4 which encodes the logits on `a,b,c,d` respectively.
* ### Induction Head Mechanism
  - Your implementation should consist of two stages.
     1. **Previous Token Head:** Identifies tokens that are directly adjacent to each other (using position information)
     2. **Induction Head:** Takes the output from the previous token head and copies the token that follows matching patterns.

     Together, these implement the pattern: `a,b,...,a -> b` for all `a, b`.
* ### Previous Token Head
  - Attend from each position to its direct predecessor
  - Extract and pass forward the token identities
  - Use `attention_strength` for the non-zero entries in the QK matrices.
  - Use only 0's and 1's in the OV matrices.
* ### Induction Head
  - Take input from the previous token head summed with the original token embedding. To make life simpler:
     - Exclude the positional embedding from this sum! That is, just sum the non-positional part of the input embedding with the output from the previous token head.
     - Delete from this sum the first token position, since the previous token head does strange things (e.g., if you have an array of `[seq_len, d_model]`, it should become `[seq_len - 1, d_model]`).
  - Copy the token that follows the previous instance of the present token.
  - Use `attention_strength` for the non-zero entries in the QK matrices.
  - Use only 0's and 1's in the OV matrices.

### Output Format:
* Return an array of 4 logits (one for each vocabulary token)
* The output should be the prediction of the induction head on the last token in the sequence. The output should be just the output from that head (**not** the sum of residual stream with the head's output!).

### Important Implementation Notes
* **Causal masking:** When implementing attention, mask out positions that come after the current position by setting their attention scores to negative infinity (e.g., `-1e9`) before applying softmax.
* **Matrix construction:** Use `attention_strength` for non-zero entries in QK matrices, and use `1.0` for non-zero entries in OV matrices (not `attention_strength`).

### Example
For the sequence `[a, b, c, d, a]`, the prediction should upweight `b`.

#### Step-by-step for this example:
**Previous Token Head** creates outputs where position $i$ contains information about the token at $i - 1$.

After deleting position 0 and adding to token embeddings (without position info), we have representations that know both "what token is here" and "what token came before"

**Induction Head** at the final position sees token `a` and looks for where else `a` appeared with its predecessor information

It finds that position 0 had token `a` (with no predecessor)

It copies what came after position 0, which is token `b`

The output logits should thus have the highest value for token `b`

### Note:
* You should your use your previously implemented function `single_attention_head` for this problem.

### Hints
- Build the previous token head and the copy head separately; test each intermediate tensor to sanity-check shapes.
- Use the positional slots (dims 4-8) to aim queries at the previous position, and the token slots (dims 0-3) to copy token identities.
- When adding the residual stream, drop position 0 so the induction head only sees valid previous-token pairs.

In [ ]:
# Embedding configuration constants
VOCAB_SIZE: int = 4  # Tokens: a, b, c, d
MAX_SEQ_LEN: int = 5  # Maximum sequence length
D_MODEL: int = 9  # 4 (vocab) + 5 (position) = 9

# Embedding dimension slices
TOKEN_DIMS = slice(0, VOCAB_SIZE)  # dims 0-3: token identity
POSITION_DIMS = slice(VOCAB_SIZE, D_MODEL)  # dims 4-8: position encoding
PREV_TOKEN_DIMS = slice(VOCAB_SIZE, 2 * VOCAB_SIZE)  # dims 4-7: previous token info

In [ ]:
def build_previous_token_head_matrices(
    seq_len: int,
    d_model: int,
    vocab_size: int,
    attention_strength: float,
) -> tuple[FloatArray, FloatArray]:
    """Build weight matrices for the previous-token attention head.

    The previous-token head attends from position i to position i-1,
    copying the previous token's identity into the output.

    Parameters
    ----------
    seq_len : int
        Length of the input sequence.
    d_model : int
        Dimension of the model embedding.
    vocab_size : int
        Size of the token vocabulary.
    attention_strength : float
        Strength of attention weights in QK matrix.

    Returns
    -------
    tuple[FloatArray, FloatArray]
        (W_QK, W_OV) matrices for the previous-token head.
    """
    w_qk = np.zeros((d_model, d_model), dtype=np.float64)
    w_ov = np.zeros((d_model, d_model), dtype=np.float64)

    # QK matrix: position i queries position i-1
    # Position i has encoding at dim (vocab_size + i)
    for pos in range(1, seq_len):
        query_dim = vocab_size + pos
        key_dim = vocab_size + (pos - 1)
        w_qk[query_dim, key_dim] = attention_strength

    # OV matrix: copy token identity to "previous token" dims
    # Maps token dim i -> dim (vocab_size + i)
    for token_idx in range(vocab_size):
        output_dim = vocab_size + token_idx
        w_ov[output_dim, token_idx] = 1.0

    return w_qk, w_ov


def build_copy_head_matrices(
    d_model: int,
    vocab_size: int,
    attention_strength: float,
) -> tuple[FloatArray, FloatArray]:
    """Build weight matrices for the copying attention head.

    The copy head matches current token (dims 0-3) against previous
    token info (dims 4-7), then copies the current token identity.

    Parameters
    ----------
    d_model : int
        Dimension of the model embedding.
    vocab_size : int
        Size of the token vocabulary.
    attention_strength : float
        Strength of attention weights in QK matrix.

    Returns
    -------
    tuple[FloatArray, FloatArray]
        (W_QK, W_OV) matrices for the copy head.
    """
    w_qk = np.zeros((d_model, d_model), dtype=np.float64)
    w_ov = np.zeros((d_model, d_model), dtype=np.float64)

    # QK matrix: query on current token matches key on previous token
    # Current token at dim i, previous token at dim (vocab_size + i)
    for token_idx in range(vocab_size):
        query_dim = token_idx
        key_dim = vocab_size + token_idx
        w_qk[query_dim, key_dim] = attention_strength

    # OV matrix: copy current token identity
    for token_idx in range(vocab_size):
        w_ov[token_idx, token_idx] = 1.0

    return w_qk, w_ov

In [ ]:
def induction_copy_head(
    embeddings: Matrix,
    attention_strength: float,
) -> FloatArray:
    """Implement an induction head for in-context pattern completion.

    The induction head predicts that patterns like `[a, b, ..., a]` will
    be followed by `b`. It consists of two attention heads:

    1. **Previous Token Head**: Creates representations containing
       information about the preceding token at each position.
    2. **Copy Head**: Finds positions with matching (current, previous)
       token patterns and copies what follows.

    Parameters
    ----------
    embeddings : Matrix
        2-hot encoded embeddings of shape (seq_len, d_model).
        Dims 0-3: token one-hot encoding (a, b, c, d).
        Dims 4-8: position one-hot encoding (0-4).
    attention_strength : float
        Scaling factor for attention logits. Higher values create
        sharper attention distributions.

    Returns
    -------
    FloatArray
        Logits for next token prediction, shape (vocab_size,).
        Higher values indicate higher probability for that token.

    Notes
    -----
    The algorithm proceeds as follows:

    1. Run previous-token head to get representations with predecessor info
    2. Add residual connection (token identity, excluding position)
    3. Drop position 0 (no valid predecessor)
    4. Run copy head to find matching patterns and copy following token
    5. Return logits for the final position

    Examples
    --------
    >>> # Sequence [a, b, c, d, a] should predict b
    >>> emb = [[1,0,0,0, 1,0,0,0,0],  # a at pos 0
    ...        [0,1,0,0, 0,1,0,0,0],  # b at pos 1
    ...        [0,0,1,0, 0,0,1,0,0],  # c at pos 2
    ...        [0,0,0,1, 0,0,0,1,0],  # d at pos 3
    ...        [1,0,0,0, 0,0,0,0,1]]  # a at pos 4
    >>> logits = induction_copy_head(emb, attention_strength=10.0)
    >>> np.argmax(logits)  # Should be 1 (token b)
    1
    """
    # Convert to numpy and extract dimensions
    input_embeddings = np.asarray(embeddings, dtype=np.float64)
    seq_len, d_model = input_embeddings.shape

    # Stage 1: Previous Token Head
    # This head attends to the previous position and copies token identity
    w_qk_prev, w_ov_prev = build_previous_token_head_matrices(
        seq_len=seq_len,
        d_model=d_model,
        vocab_size=VOCAB_SIZE,
        attention_strength=attention_strength,
    )

    prev_token_output = single_attention_head(
        input_embeddings, w_qk_prev, w_ov_prev
    )

    # Add residual connection: combine with original token embeddings
    # Exclude positional encoding from residual (only token identity)
    combined_representation = prev_token_output.copy()
    combined_representation[:, TOKEN_DIMS] += input_embeddings[:, TOKEN_DIMS]

    # Drop position 0 (previous-token head has no valid predecessor there)
    combined_representation = combined_representation[1:, :]

    # Stage 2: Copy Head
    # Matches current token against previous token patterns, copies following token
    w_qk_copy, w_ov_copy = build_copy_head_matrices(
        d_model=d_model,
        vocab_size=VOCAB_SIZE,
        attention_strength=attention_strength,
    )

    copy_output = single_attention_head(
        combined_representation, w_qk_copy, w_ov_copy
    )

    # Extract logits for final position (next token prediction)
    final_logits = copy_output[-1, TOKEN_DIMS]

    return final_logits

### Test `induction_copy_head` basic

In [ ]:
embeddings = [[1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0],
              [0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0],
              [0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0],
              [0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0, 0.0],
              [1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]]
attention_strength = 10.0

out = induction_copy_head(embeddings, attention_strength).tolist()

expected_out = [0.000045, 0.999864, 0.000045, 0.000045]

assert np.isclose(expected_out, out, atol=NUMERICAL_TOLERANCE).all(), f"Failed:\nExpected: {expected_out}\nGot: {out}"
print("Test case passed ✅")

### Test `induction_copy_head`

In [ ]:
with open('induction_head_test_cases.json', 'r') as f:
    induction_head_test_cases = json.load(f)

for test_case_id in induction_head_test_cases:
    embeddings, attention_strength, expected_out = induction_head_test_cases[test_case_id]
    out = induction_copy_head(embeddings, attention_strength).tolist()
    test_id = test_case_id.split(" ")[-1]
    assert np.isclose(expected_out, out, atol=NUMERICAL_TOLERANCE).all(), f"Test Case {test_id} Failed:\nExpected: {expected_out}\nGot: {out}"

print("All test cases passed ✅")